# Мульти-продуктовый прогноз комиссий — v4

**Структура:** Шаг 0 (загрузка) → Часть 1 (прогноз → `df_forecast_combined.xlsx`) → Часть 2 (сравнение с фактом → `comparison_results.xlsx`).

**Разбор ошибок v3 и что изменено:**
1. **Загиб прогноза вниз** (занижение по ИЭ и др.): в конце train сидит сезонный
   новогодний спад (Dec'25→Jan'26 = -16%, годом ранее было то же -11%). Расширенные
   в v3 границы `changepoint_range` (до 0.98) и `changepoint_prior_scale` (до 1.0)
   позволяли модели прочитать этот СЕЗОННЫЙ провал как ПЕРЕЛОМ ТРЕНДА и тянуть спад
   на 9 месяцев вперёд. **Границы возвращены к консервативным** (0.80–0.92 / 0.001–0.5).
2. **Лог-масштаб для короткой истории вредил** (СБП AFT: 14.5% → 32.1%). Убран.
3. **Выбор масштаба (лог/лин) вынесен из Optuna** в быстрый детерминированный
   бэктест — пространство поиска для 30 trials стало вдвое компактнее.
4. **Главное: страховочный турнир на бэктесте.** По каждому продукту настроенный
   Prophet соревнуется с наивной моделью (средний уровень по дням недели за последние
   56 дней) на последних 91 дне train. В прогноз идёт победитель. Проверка на факте:
   наивная модель бьёт Prophet v3 на ИЭ (8.6% vs 15.8%), Выплатах (22.5% vs 34.4%),
   СБП NFC (11.9% vs 35.6%), но проигрывает на ТЭ и СБП ПП — поэтому выбор по-продуктовый.

**Про Optuna:** проблема была не в оптимизаторе (TPE в Optuna — сильный выбор,
random search / Hyperopt здесь ничего не добавят), а в слишком широком пространстве
поиска и в отсутствии финальной проверки модели на честном бэктесте. Оба пункта исправлены.

In [ ]:
# ============================================================
# ИМПОРТЫ И ОБЩИЕ НАСТРОЙКИ
# ============================================================
import warnings
import logging
import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

import holidays
import optuna
from prophet import Prophet
from prophet.diagnostics import cross_validation
from sklearn.metrics import r2_score, mean_absolute_error, mean_absolute_percentage_error

# бустер: LightGBM, при его отсутствии — тихий фолбэк на sklearn
try:
    from lightgbm import LGBMRegressor as BoosterModel
    BOOSTER_NAME = "LightGBM"
except ImportError:
    from sklearn.ensemble import HistGradientBoostingRegressor as BoosterModel
    BOOSTER_NAME = "HistGradientBoosting (sklearn fallback)"

# ============================================================
# ОТКЛЮЧЕНИЕ ЛОГОВ
# ------------------------------------------------------------
# cmdstanpy вешает свой хендлер и ставит уровень INFO В МОМЕНТ ИМПОРТА,
# поэтому глушим его ПОСЛЕ "from prophet import Prophet".
# Фильтр + disabled=True переживают повторный setLevel() внутри fit().
# ============================================================

class _DropAll(logging.Filter):
    def filter(self, record):
        return False          # не пропускаем ни одну запись

def silence_logger(name):
    lg = logging.getLogger(name)
    lg.handlers.clear()       # снимаем собственный хендлер библиотеки
    lg.addFilter(_DropAll())  # фильтр переживает повторный setLevel()
    lg.setLevel(logging.CRITICAL)
    lg.propagate = False
    lg.disabled = True        # добивающий выключатель
    return lg

for _name in ("cmdstanpy", "prophet", "prophet.models"):
    silence_logger(_name)

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.ERROR)

print(f"Бустер: {BOOSTER_NAME}")

## Шаг 0. Загрузка данных и разделение train / fact

In [ ]:
# ============================================================
# ШАГ 0. ЗАГРУЗКА ДАННЫХ
# ------------------------------------------------------------
# df_train — данные для обучения: 2024-01-01 ... 2026-03-31 (включительно)
# df_fact  — весь факт за доступный период (только для сравнения в Части 2,
#            в обучении НЕ участвует)
# ============================================================

FILE_PATH = "data_multi_product_fact.xlsx"   # исходный файл
TRAIN_END = pd.Timestamp("2026-03-31")        # конец обучающего периода

df_raw = pd.read_excel(FILE_PATH)
df_raw["VALUE_DAY"] = pd.to_datetime(df_raw["VALUE_DAY"])

df_train = df_raw[df_raw["VALUE_DAY"] <= TRAIN_END].copy()
df_fact  = df_raw.copy()   # факт целиком — для Части 2

print(f"Всего строк:  {len(df_raw)}")
print(f"df_train:     {len(df_train)} строк, {df_train['VALUE_DAY'].min().date()} ... {df_train['VALUE_DAY'].max().date()}")
print(f"df_fact:      {len(df_fact)} строк,  {df_fact['VALUE_DAY'].min().date()} ... {df_fact['VALUE_DAY'].max().date()}")
print(f"Продукты:     {sorted(df_train['PRODUCT'].unique())}")

## Часть 1. Прогноз по каждому продукту

Логика для каждого продукта с нормальной историей:
1. **Бэктест-окно** — последние 91 день train (обучение на остальном, проверка на окне).
2. **Выбор масштаба** (лог / линейный): два быстрых Prophet на бэктесте, берём лучший по MAPE.
3. **Optuna + кросс-валидация** в выбранном масштабе, цель — MAPE, консервативные границы changepoints.
4. **Турнир на бэктесте**: настроенный Prophet vs наивная модель (средний уровень
   по дням недели за последние 56 дней). Победитель по MAPE идёт в финал.
5. Победитель переобучается на полном train; для Prophet — **guarded-бустер** на остатках
   (включается только если улучшает MAPE на последних 60 днях train).
6. Отрицательные прогнозы обрезаются в 0.

Продукты с историей < 180 дней (СБП AFT / СБП NFC): без Optuna, укороченный бэктест 28 дней,
тот же турнир «простой Prophet vs наивная модель».

In [ ]:
# ============================================================
# ЧАСТЬ 1 / НАСТРОЙКИ
# ============================================================

N_TRIALS          = 30    # число попыток Optuna на продукт
FORECAST_END      = "2026-12-31"
MIN_HISTORY_DAYS  = 180   # порог короткой истории
HOLDOUT_DAYS      = 60    # хвост train для проверки бустера
BACKTEST_DAYS     = 91    # бэктест-окно (для короткой истории — 28)
NAIVE_WINDOW      = 56    # окно наивной модели, дней

# --- российские праздники для Prophet ---
years = list(range(2024, 2027))
ru_holidays = holidays.Russia(years=years)
holidays_df = pd.DataFrame([
    {"ds": pd.to_datetime(dt), "holiday": name, "lower_window": 0, "upper_window": 3}
    for dt, name in ru_holidays.items()
])

# --- сезонные регрессоры: лето и зима ---
def add_regressors(df):
    df = df.copy()
    df["summer_season"] = df["ds"].dt.month.isin([6, 7, 8]).astype(int)
    df["winter_season"] = df["ds"].dt.month.isin([12, 1, 2]).astype(int)
    return df

# --- наивная модель-бенчмарк: средний уровень по дням недели за последние 56 дней ---
def naive_forecast(dfp_hist, dates):
    tail = dfp_hist.tail(NAIVE_WINDOW)
    dow_mean = tail.groupby(tail["ds"].dt.dayofweek)["y"].mean()
    return dates.dt.dayofweek.map(dow_mean).values

# --- календарные признаки для бустера (только циклические, без тренда) ---
def calendar_features(ds_series):
    X = pd.DataFrame(index=ds_series.index)
    X["dow"]        = ds_series.dt.dayofweek
    X["day"]        = ds_series.dt.day
    X["month"]      = ds_series.dt.month
    X["weekofyear"] = ds_series.dt.isocalendar().week.astype(int)
    X["is_holiday"] = ds_series.isin(holidays_df["ds"]).astype(int)
    X["summer"]     = ds_series.dt.month.isin([6, 7, 8]).astype(int)
    X["winter"]     = ds_series.dt.month.isin([12, 1, 2]).astype(int)
    return X

In [ ]:
# ============================================================
# ЧАСТЬ 1 / ОСНОВНОЙ ЦИКЛ ПО ПРОДУКТАМ
# ============================================================

products = sorted(df_train["PRODUCT"].unique())
all_forecasts = []   # прогнозы по каждому продукту
tuning_log    = []   # краткий лог: масштаб, чемпион, бустер, параметры

for product in products:
    print(f"▶ {product}: tuning + forecast ...", end=" ", flush=True)

    # ---------- 1. подготовка ряда ----------
    dfp = (df_train[df_train["PRODUCT"] == product]
           .rename(columns={"VALUE_DAY": "ds", "FEE_RUR_AMT_SUM": "y"})
           [["ds", "y"]]
           .sort_values("ds")
           .reset_index(drop=True))
    y_raw = dfp["y"].copy()

    history_days  = (dfp["ds"].max() - dfp["ds"].min()).days
    short_history = history_days < MIN_HISTORY_DAYS
    bt_days       = 28 if short_history else BACKTEST_DAYS

    future = pd.DataFrame({"ds": pd.date_range(dfp["ds"].min(), FORECAST_END, freq="D")})

    # ---------- 2. бэктест-окно: последние bt_days дней train ----------
    bt_cut   = dfp["ds"].max() - pd.Timedelta(days=bt_days)
    dfp_sub  = dfp[dfp["ds"] <= bt_cut].reset_index(drop=True)   # обучение для бэктеста
    dfp_test = dfp[dfp["ds"] >  bt_cut].reset_index(drop=True)   # проверка

    dfp_reg     = add_regressors(dfp)
    dfp_sub_reg = add_regressors(dfp_sub)
    future_reg  = add_regressors(future)
    test_reg    = add_regressors(dfp_test[["ds"]].copy())

    best_params, booster_used, use_log = {}, False, False

    if short_history:
        # ---------- упрощённая ветка: простой Prophet (линейный масштаб!) ----------
        def make_model():
            m = Prophet(holidays=holidays_df, weekly_seasonality=True,
                        yearly_seasonality=False, daily_seasonality=False,
                        interval_width=0.95)
            m.add_seasonality(name="monthly", period=30.5, fourier_order=5)
            return m

        m_bt = make_model()
        m_bt.fit(dfp_sub)
        p_bt = m_bt.predict(dfp_test[["ds"]])["yhat"].clip(lower=0)
        mape_prophet_bt = mean_absolute_percentage_error(dfp_test["y"], p_bt)

    else:
        # ---------- 3. выбор масштаба (лог vs линейный) на бэктесте ----------
        # лог хорош для мультипликативного роста (СБП ПП +120% г/г),
        # линейный — когда тренд ломается или выходит на плато
        def quick_predict(use_log_flag):
            m = Prophet(holidays=holidays_df, weekly_seasonality=True,
                        yearly_seasonality=False, daily_seasonality=False,
                        changepoint_prior_scale=0.1, interval_width=0.95)
            m.add_seasonality(name="yearly", period=365.25, fourier_order=8)
            m.add_regressor("summer_season"); m.add_regressor("winter_season")
            d = dfp_sub_reg.copy()
            if use_log_flag:
                d["y"] = np.log1p(d["y"])
            m.fit(d)
            p = m.predict(test_reg)["yhat"]
            return (np.expm1(p) if use_log_flag else p).clip(lower=0)

        mape_lin = mean_absolute_percentage_error(dfp_test["y"], quick_predict(False))
        mape_log = mean_absolute_percentage_error(dfp_test["y"], quick_predict(True))
        use_log  = mape_log < mape_lin

        # ---------- 4. Optuna: подбор гиперпараметров через CV, цель — MAPE ----------
        def objective(trial):
            params = {
                # консервативные границы: не даём сезонному январскому спаду
                # в конце train превратиться в "перелом тренда"
                "changepoint_prior_scale": trial.suggest_float("changepoint_prior_scale", 0.001, 0.5, log=True),
                "changepoint_range":       trial.suggest_float("changepoint_range", 0.80, 0.92),
                "n_changepoints":          trial.suggest_int("n_changepoints", 10, 40),
                "seasonality_prior_scale": trial.suggest_float("seasonality_prior_scale", 0.01, 10.0, log=True),
                "holidays_prior_scale":    trial.suggest_float("holidays_prior_scale", 0.01, 10.0, log=True),
            }
            # в логах сезонность аддитивна по построению, в линейном масштабе — подбираем
            params["seasonality_mode"] = ("additive" if use_log else
                trial.suggest_categorical("seasonality_mode", ["additive", "multiplicative"]))

            m = Prophet(holidays=holidays_df, weekly_seasonality=True,
                        yearly_seasonality=False, daily_seasonality=False,
                        interval_width=0.95, **params)
            m.add_seasonality(name="yearly", period=365.25,
                              fourier_order=trial.suggest_int("yearly_fourier", 3, 12))
            m.add_regressor("summer_season")
            m.add_regressor("winter_season")

            df_fit = dfp_reg.copy()
            if use_log:
                df_fit["y"] = np.log1p(df_fit["y"])
            m.fit(df_fit)

            df_cv = cross_validation(m, initial="365 days", period="61 days",
                                     horizon="91 days", parallel=None,
                                     disable_tqdm=True)
            # MAPE всегда в исходном масштабе (рубли)
            y_true = np.expm1(df_cv["y"])    if use_log else df_cv["y"]
            y_pred = np.expm1(df_cv["yhat"]) if use_log else df_cv["yhat"]
            return mean_absolute_percentage_error(y_true, y_pred)

        study = optuna.create_study(direction="minimize")   # MAPE -> минимизация
        study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)
        best_params    = study.best_params.copy()
        yearly_fourier = best_params.pop("yearly_fourier")
        if use_log:
            best_params["seasonality_mode"] = "additive"

        def make_model():
            m = Prophet(holidays=holidays_df, weekly_seasonality=True,
                        yearly_seasonality=False, daily_seasonality=False,
                        interval_width=0.95, **best_params)
            m.add_seasonality(name="yearly", period=365.25, fourier_order=yearly_fourier)
            m.add_regressor("summer_season")
            m.add_regressor("winter_season")
            return m

        # бэктест настроенного Prophet: обучение без последних 91 дня, проверка на них
        m_bt = make_model()
        d = dfp_sub_reg.copy()
        if use_log:
            d["y"] = np.log1p(d["y"])
        m_bt.fit(d)
        p_bt = m_bt.predict(test_reg)["yhat"]
        if use_log:
            p_bt = np.expm1(p_bt)
        p_bt = p_bt.clip(lower=0)
        mape_prophet_bt = mean_absolute_percentage_error(dfp_test["y"], p_bt)

    # ---------- 5. ТУРНИР на бэктесте: Prophet vs наивная модель ----------
    naive_bt      = naive_forecast(dfp_sub, dfp_test["ds"])
    mape_naive_bt = mean_absolute_percentage_error(dfp_test["y"], naive_bt)
    champion      = "prophet" if mape_prophet_bt <= mape_naive_bt else "naive"

    # ---------- 6. финальный прогноз победителя (обучение на ПОЛНОМ train) ----------
    if champion == "naive":
        # наивная модель: профиль дней недели за последние 56 дней train
        fc = pd.DataFrame({"ds": future["ds"],
                           "yhat": naive_forecast(dfp, future["ds"])})
        yhat_final = fc["yhat"].clip(lower=0)

    else:
        model = make_model()
        if short_history:
            model.fit(dfp)
            fc = model.predict(future)
        else:
            df_fit = dfp_reg.copy()
            if use_log:
                df_fit["y"] = np.log1p(df_fit["y"])
            model.fit(df_fit)
            fc = model.predict(future_reg)
            if use_log:
                fc["yhat"] = np.expm1(fc["yhat"])

        # ---------- guarded-бустер на остатках (в исходном масштабе) ----------
        if not short_history:
            fc_train  = fc[fc["ds"].isin(dfp["ds"])].reset_index(drop=True)
            residuals = y_raw.values - fc_train["yhat"].values

            cut   = len(dfp) - HOLDOUT_DAYS     # [обучение бустера | 60 дней — проверка]
            X_all = calendar_features(dfp["ds"])
            booster = BoosterModel(random_state=42)
            booster.fit(X_all.iloc[:cut], residuals[:cut])

            res_pred_holdout = booster.predict(X_all.iloc[cut:])
            mape_prophet = mean_absolute_percentage_error(y_raw.iloc[cut:], fc_train["yhat"].iloc[cut:])
            mape_boosted = mean_absolute_percentage_error(
                y_raw.iloc[cut:], (fc_train["yhat"].iloc[cut:] + res_pred_holdout).clip(lower=0))
            booster_used = mape_boosted < mape_prophet

            if booster_used:
                booster.fit(X_all, residuals)   # переобучаем на всех остатках
                fc["yhat"] = fc["yhat"] + booster.predict(calendar_features(fc["ds"]))

        yhat_final = fc["yhat"].clip(lower=0)   # комиссии не бывают отрицательными

    print(f"готово (backtest: Prophet {mape_prophet_bt*100:.1f}% vs naive {mape_naive_bt*100:.1f}% "
          f"-> {champion}, масштаб {'log' if use_log else 'lin'}, бустер {'ON' if booster_used else 'off'}"
          + (", короткая история" if short_history else "") + ")")

    # ---------- сборка результата по продукту ----------
    all_forecasts.append(pd.DataFrame({
        "ds":        fc["ds"],
        "PRODUCT":   product,
        "yhat":      yhat_final,
        "train_end": TRAIN_END,
    }))
    tuning_log.append({"PRODUCT": product,
                       "short_history": short_history,
                       "champion": champion,
                       "use_log": use_log,
                       "booster": booster_used,
                       "mape_bt_prophet_%": round(mape_prophet_bt * 100, 1),
                       "mape_bt_naive_%":   round(mape_naive_bt * 100, 1),
                       **best_params})

print("\nГотово: прогнозы по всем продуктам построены.")

In [ ]:
# ============================================================
# ЧАСТЬ 1 / ИТОГ: единый датасет прогноза по дням и продуктам
# ============================================================

df_forecast_combined = pd.concat(all_forecasts, ignore_index=True)
df_forecast_combined.to_excel("df_forecast_combined.xlsx", index=False)

print(f"df_forecast_combined: {len(df_forecast_combined)} строк -> df_forecast_combined.xlsx")
display(pd.DataFrame(tuning_log))
df_forecast_combined.head()

## Часть 2. Сравнение прогноза с фактом

Часть 2 **независима**: читает `df_forecast_combined.xlsx` и исходный факт из Excel.

Метрики считаются **только на out-of-sample периоде** (после `train_end`): R², MAE и **MAPE**.
На каждый продукт — три панели: дневная динамика, помесячные суммы, гистограмма ошибок OOS с метриками.

In [ ]:
# ============================================================
# ЧАСТЬ 2 / ЗАГРУЗКА И ОБЪЕДИНЕНИЕ
# ============================================================

FORECAST_FILE = "df_forecast_combined.xlsx"
FACT_FILE     = "data_multi_product_fact.xlsx"

df_fc = pd.read_excel(FORECAST_FILE)
df_fc["ds"]        = pd.to_datetime(df_fc["ds"])
df_fc["train_end"] = pd.to_datetime(df_fc["train_end"])

df_fact2 = pd.read_excel(FACT_FILE)
df_fact2["VALUE_DAY"] = pd.to_datetime(df_fact2["VALUE_DAY"])
df_fact2 = df_fact2.rename(columns={"VALUE_DAY": "ds", "FEE_RUR_AMT_SUM": "y_fact"})

# объединяем прогноз и факт по дате и продукту (факт есть не на все дни прогноза)
df_cmp = df_fc.merge(df_fact2[["ds", "PRODUCT", "y_fact"]], on=["ds", "PRODUCT"], how="left")

# флаг out-of-sample: день после конца обучения И по нему есть факт
df_cmp["is_oos"] = (df_cmp["ds"] > df_cmp["train_end"]) & df_cmp["y_fact"].notna()

print(f"Строк всего: {len(df_cmp)}, из них OOS с фактом: {df_cmp['is_oos'].sum()}")
print(f"OOS период: {df_cmp.loc[df_cmp['is_oos'], 'ds'].min().date()} ... {df_cmp.loc[df_cmp['is_oos'], 'ds'].max().date()}")

In [ ]:
# ============================================================
# ЧАСТЬ 2 / МЕТРИКИ ПО ПРОДУКТАМ (только OOS!)
# ============================================================

metrics_rows = []
for product, g in df_cmp[df_cmp["is_oos"]].groupby("PRODUCT"):
    y_true, y_pred = g["y_fact"].values, g["yhat"].values
    metrics_rows.append({
        "PRODUCT":  product,
        "days_oos": len(g),
        "R2":       r2_score(y_true, y_pred),
        "MAE":      mean_absolute_error(y_true, y_pred),
        "MAPE_%":   mean_absolute_percentage_error(y_true, y_pred) * 100,
        "fact_sum":     y_true.sum(),
        "forecast_sum": y_pred.sum(),
        "diff_%":   (y_pred.sum() / y_true.sum() - 1) * 100,   # перекос суммарного прогноза
    })

df_metrics = (pd.DataFrame(metrics_rows)
              .sort_values("MAPE_%")
              .reset_index(drop=True))

# сохраняем результаты сравнения: дневные данные + метрики
with pd.ExcelWriter("comparison_results.xlsx") as writer:
    df_cmp.to_excel(writer, sheet_name="daily", index=False)
    df_metrics.to_excel(writer, sheet_name="metrics_oos", index=False)

print("Сохранено -> comparison_results.xlsx")
display(df_metrics.style.format({"R2": "{:.3f}", "MAE": "{:,.0f}", "MAPE_%": "{:.1f}",
                                 "fact_sum": "{:,.0f}", "forecast_sum": "{:,.0f}", "diff_%": "{:+.1f}"}))

In [ ]:
# ============================================================
# ЧАСТЬ 2 / ГРАФИКИ: по 3 панели на продукт
#   1) дневная динамика: факт vs прогноз (OOS зона выделена)
#   2) помесячные суммы с подписями значений
#   3) гистограмма дневных ошибок OOS + блок метрик
# ============================================================

today_month = df_fact2["ds"].max().to_period("M")   # текущий (неполный) месяц — исключаем из факта

for product in df_metrics["PRODUCT"]:
    g = df_cmp[df_cmp["PRODUCT"] == product].sort_values("ds")
    train_end = g["train_end"].iloc[0]
    m = df_metrics[df_metrics["PRODUCT"] == product].iloc[0]

    fig, axes = plt.subplots(1, 3, figsize=(22, 5.5))
    fig.suptitle(f"{product}", fontsize=14, fontweight="bold")

    # ---------- панель 1: дневная динамика ----------
    ax = axes[0]
    ax.plot(g["ds"], g["y_fact"], label="факт", color="black", lw=0.9)
    ax.plot(g["ds"], g["yhat"], label="прогноз", color="tab:blue", lw=0.9, alpha=0.85)
    ax.axvline(train_end, color="red", ls="--", lw=1, label="конец train")
    ax.axvspan(train_end, g["ds"].max(), color="orange", alpha=0.08)
    ax.set_title("Дневная динамика")
    ax.legend(loc="upper left")
    ax.grid(alpha=0.3)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    ax.tick_params(axis="x", rotation=45)

    # ---------- панель 2: помесячные суммы ----------
    gm = g.copy()
    gm["ym"] = gm["ds"].dt.to_period("M")
    monthly_fc   = gm.groupby("ym")["yhat"].sum()
    monthly_fact = gm[gm["ym"] != today_month].groupby("ym")["y_fact"].sum(min_count=1)  # неполный месяц убираем

    ax = axes[1]
    x_fc = monthly_fc.index.to_timestamp()
    ax.plot(x_fc, monthly_fc.values, label="прогноз", marker="s", color="tab:blue", zorder=2)
    mask = monthly_fact.notna() & (monthly_fact > 0)
    x_fact = monthly_fact[mask].index.to_timestamp()
    ax.plot(x_fact, monthly_fact[mask].values, label="факт", marker="o", color="black", zorder=3)

    # подписи: прогноз сверху (жирнее), факт снизу и только там где есть данные
    for x, v in zip(x_fc, monthly_fc.values):
        ax.annotate(f"{v/1e9:.1f}", (x, v), textcoords="offset points", xytext=(0, 10),
                    ha="center", fontsize=8, fontweight="bold", color="tab:blue", zorder=5)
    for x, v in zip(x_fact, monthly_fact[mask].values):
        ax.annotate(f"{v/1e9:.1f}", (x, v), textcoords="offset points", xytext=(0, -15),
                    ha="center", fontsize=8, color="black", zorder=5)

    ax.axvline(train_end, color="red", ls="--", lw=1)
    ax.set_title("Помесячные суммы, млрд руб.")
    ax.legend(loc="upper left")
    ax.grid(alpha=0.3)
    ax.tick_params(axis="x", rotation=45)

    # ---------- панель 3: ошибки OOS ----------
    ax = axes[2]
    oos = g[g["is_oos"]]
    err_pct = (oos["yhat"] - oos["y_fact"]) / oos["y_fact"] * 100
    ax.hist(err_pct, bins=25, color="tab:blue", alpha=0.7, edgecolor="white")
    ax.axvline(0, color="black", lw=1)
    ax.axvline(err_pct.mean(), color="red", ls="--", lw=1, label=f"средняя: {err_pct.mean():+.1f}%")
    ax.set_title("Дневные ошибки OOS, % от факта")
    ax.legend(loc="upper left")
    ax.grid(alpha=0.3)

    # блок метрик
    txt = (f"R²    = {m['R2']:.3f}\n"
           f"MAE  = {m['MAE']/1e6:,.1f} млн\n"
           f"MAPE = {m['MAPE_%']:.1f}%\n"
           f"Сумма: {m['diff_%']:+.1f}%")
    ax.text(0.97, 0.97, txt, transform=ax.transAxes, va="top", ha="right",
            fontsize=10, family="monospace",
            bbox=dict(boxstyle="round", facecolor="lightyellow", alpha=0.9))

    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# ЧАСТЬ 2 / КРАТКИЙ ВЫВОД
# ============================================================

print("Итог по OOS периоду (сортировка по MAPE):\n")
for _, r in df_metrics.iterrows():
    flag = "OK " if r["MAPE_%"] < 15 else ("~  " if r["MAPE_%"] < 30 else "!! ")
    print(f"  {flag}{r['PRODUCT']:<26} MAPE={r['MAPE_%']:5.1f}%  R2={r['R2']:6.3f}  "
          f"перекос суммы {r['diff_%']:+.1f}%")

total_fact = df_metrics["fact_sum"].sum()
total_fc   = df_metrics["forecast_sum"].sum()
print(f"\nВесь портфель: факт {total_fact/1e9:,.1f} млрд vs прогноз {total_fc/1e9:,.1f} млрд "
      f"({(total_fc/total_fact-1)*100:+.1f}%)")